# ⚡ Inference Pipeline: ISO-NE Energy Demand Forecasting

## 🎯 Overview
This notebook simulates the **production inference phase** of our MLOps pipeline. In the real world, grid operators cannot rely on static historical CSVs; they must fetch live telemetry and weather forecasts to predict electricity demand hours or days in advance to balance the grid and prevent blackouts. 

Here, we will load our saved, production-ready artifacts (the Scikit-Learn preprocessor and the Champion Model) and generate predictions for unseen, future timestamps.

## 🌐 Data Acquisition Strategy (The APIs)
Because electricity demand is driven heavily by human routines and thermodynamics, our automated inference pipeline must stitch together data from two distinct external sources:

### 1. The Weather API (Meteorological Forecasts)
We need the forecasted Temperature (`Dry_Bulb`) and Humidity (`Dew_Point`) for the exact hours we are trying to predict.
* **Source:** Open-Meteo (Free, no API key required, highly reliable for grid operations)
* **Documentation:** [Open-Meteo Forecast API](https://open-meteo.com/en/docs)
* **What we fetch:** Hourly temperature and dew point forecasts for the New England coordinates.

### 2. The Grid & Calendar Data (ISO-NE)
We need the exact calendar timestamps (to generate cyclical time features and holiday flags) and recent grid telemetry.
* **Source:** ISO New England (ISO Express Portal)
* **Portal Link:** [ISO-NE ISO Express](https://www.iso-ne.com/isoexpress/)
* **What we fetch:** Target dates, hour endings, and recent real-time system load 
---

## 🔄 The Inference Workflow
1. **Fetch:** Query the Weather API and ISO-NE for the target forecast window (e.g., the next 24 hours for Day-Ahead prediction).
2. **Merge:** Join the weather forecasts with the calendar timestamps into a single, raw Pandas DataFrame.
3. **Transform:** Pass the raw data through our saved `preprocessor.pkl` (which automatically handles datetime extraction, holiday flags, thermodynamic U-curves, and RobustScaling).
4. **Predict:** Pass the transformed matrix into our saved `best_model.pkl` to output the predicted System Load in Megawatts (MW).
5. **Evaluate:** Compare the predictions against the actual "ground truth" (once the hour passes) to calculate MAPE and monitor for data drift.

In [48]:
import requests
import pandas as pd
import time
from dataclasses import dataclass, field
from requests.auth import HTTPBasicAuth


In [49]:
@dataclass
class APIConfig:
    """Configurations for the Open-Meteo Historical API."""
    # Date components
    year: int = 2025
    month: int = 9
    day: int = 10
    
    # We will let Python generate this automatically
    formatted_date: str = field(init=False)
    
    # API Settings
    latitude: float = 42.3601
    longitude: float = -71.0589
    hourly_vars: str = "temperature_2m,dew_point_2m"
    temp_unit: str = "fahrenheit"
    timezone: str = "America/New_York"
    max_retries: int = 3
    backoff_sec: int = 3
    url: str = "https://archive-api.open-meteo.com/v1/archive"

    """Configurations for the ISO-NE Web Services API."""
    # Replace with your verified ISO Express login
    email: str = "abdelhadiosama12@gmail.com"
    password: str = "Abdoo12dy"

    
    def __post_init__(self):
        """Automatically builds the strict YYYY-MM-DD string the API requires."""
        # The :02d ensures single digits like 5 become "05"
        self.formatted_date = f"{self.year}-{self.month:02d}-{self.day:02d}"

config = APIConfig()

### 1. The Weather API

In [50]:
def fetch_historical_weather(config) -> pd.DataFrame:
    """Fetches 24 hours of historical weather data using the config's date."""
    params = {
        "latitude": config.latitude,
        "longitude": config.longitude,
        "start_date": config.formatted_date,
        "end_date": config.formatted_date,
        "hourly": config.hourly_vars,
        "temperature_unit": config.temp_unit,
        "timezone": config.timezone
    }
    
    for attempt in range(1, config.max_retries + 1):
        try:
            response = requests.get(config.url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
            
            df_weather = pd.DataFrame({
                'Date': pd.to_datetime(data['hourly']['time']),
                'Dry_Bulb': data['hourly']['temperature_2m'],
                'Dew_Point': data['hourly']['dew_point_2m']
            })
            
            # Shift Open-Meteo's Hr_Start (0-23) to ISO-NE's Hr_End (1-24)
            df_weather['Hr_End'] = df_weather['Date'].dt.hour + 1
            df_weather['Date'] = df_weather['Date'].dt.date
            
            print(f"✅ Weather fetched successfully for {config.formatted_date}")
            return df_weather
            
        except requests.RequestException as e:
            print(f"⚠️ Attempt {attempt}/{config.max_retries} failed: {e}")
            if attempt < config.max_retries:
                time.sleep(config.backoff_sec * attempt)
                
    raise ConnectionError(f"❌ Failed to fetch weather after {config.max_retries} attempts.")

# Test the execution

df_weather_hist = fetch_historical_weather(config)
display(df_weather_hist)

✅ Weather fetched successfully for 2025-09-10


,Date,Dry_Bulb,Dew_Point,Hr_End
0,2025-09-10,58.8,55.9,1
1,2025-09-10,58.6,55.6,2
2,2025-09-10,58.4,55.3,3
3,2025-09-10,58.5,55.2,4
4,2025-09-10,58.3,55.1,5
5,2025-09-10,58.2,55.3,6
6,2025-09-10,57.7,55.2,7
7,2025-09-10,58.0,55.4,8
8,2025-09-10,59.5,56.5,9
9,2025-09-10,61.2,56.8,10


In [51]:
import requests
import pandas as pd
import time
from dataclasses import dataclass, field
from requests.auth import HTTPBasicAuth

@dataclass
class GridAPIConfig:
    """Configurations for the ISO-NE Web Services API."""
    year: int = 2024
    month: int = 5
    day: int = 15
    
    # Replace with your verified ISO Express login
    email: str = "your_email@example.com"
    password: str = "your_password"
    
    formatted_date: str = field(init=False)
    url: str = field(init=False)
    max_retries: int = 3
    backoff_sec: int = 3
    
    def __post_init__(self):
        # 1. Format dates strictly as YYYYMMDD (e.g., 20240515)
        self.formatted_date = f"{self.year}{self.month:02d}{self.day:02d}"
        # 2. Inject the date into the historical day endpoint
        self.url = f"https://webservices.iso-ne.com/api/v1.1/hourlysysload/day/{self.formatted_date}.json"


def fetch_historical_load(config: GridAPIConfig) -> pd.DataFrame:
    """Fetches 24 hours of actual System Load from ISO-NE."""
    for attempt in range(1, config.max_retries + 1):
        try:
            response = requests.get(
                config.url, 
                auth=HTTPBasicAuth(config.email, config.password),
                timeout=10
            )
            response.raise_for_status()
            data = response.json()
            
            # Extract the nested list of hourly records
            records = data.get('HourlySystemLoads', {}).get('HourlySystemLoad', [])
            
            if not records:
                raise ValueError(f"API returned empty data for {config.formatted_date}.")
                
            df_grid = pd.DataFrame(records)
            
            # Format columns to perfectly match our weather dataframe and training schema
            df_grid = df_grid[['BeginDate', 'Load']].copy()
            df_grid = df_grid.rename(columns={'Load': 'System_Load'})
            
            df_grid['BeginDate'] = pd.to_datetime(df_grid['BeginDate'])
            df_grid['Hr_End'] = df_grid['BeginDate'].dt.hour + 1
            df_grid['Date'] = df_grid['BeginDate'].dt.date
            df_grid = df_grid.drop(columns=['BeginDate'])
            
            print(f"✅ Grid load fetched successfully for {config.formatted_date}")
            return df_grid
            
        except requests.RequestException as e:
            print(f"⚠️ Attempt {attempt}/{config.max_retries} failed: {e}")
            if attempt < config.max_retries:
                time.sleep(config.backoff_sec * attempt)
                
    raise ConnectionError(f"❌ Failed to fetch grid data after {config.max_retries} attempts.")

# Execution
grid_config = GridAPIConfig(year=config.year, month=config.month, day=config.day, email="abdelhadiosama12@gmail.com", password="Abdoo12dy")
df_grid_hist = fetch_historical_load(config=grid_config)
display(df_grid_hist)

✅ Grid load fetched successfully for 20250910


,System_Load,Hr_End,Date
0,10728.664,1,2025-09-10
1,10619.604,2,2025-09-10
2,10392.207,3,2025-09-10
3,10333.420,4,2025-09-10
4,10528.720,5,2025-09-10
5,10979.342,6,2025-09-10
6,11880.971,7,2025-09-10
7,12165.776,8,2025-09-10
8,11952.447,9,2025-09-10
9,11298.770,10,2025-09-10


In [52]:
import pandas as pd

def merge_weather_and_grid(df_weather: pd.DataFrame, df_grid: pd.DataFrame) -> pd.DataFrame:
    """
    Merges Open-Meteo weather and ISO-NE system load data on Date and Hr_End,
    matching the exact historical schema.
    """
    # Standardize join key types to avoid type mismatch errors
    df_weather['Date'] = pd.to_datetime(df_weather['Date']).dt.date
    df_grid['Date'] = pd.to_datetime(df_grid['Date']).dt.date
    
    df_weather['Hr_End'] = df_weather['Hr_End'].astype(int)
    df_grid['Hr_End'] = df_grid['Hr_End'].astype(int)

    # Merge on chronological keys
    df_merged = pd.merge(df_grid, df_weather, on=['Date', 'Hr_End'], how='inner')

    # Enforce exact column order
    ordered_cols = ['Date', 'Hr_End', 'System_Load', 'Dry_Bulb', 'Dew_Point']
    df_merged = df_merged[ordered_cols]

    # Chronological sort
    df_merged = df_merged.sort_values(by=['Date', 'Hr_End']).reset_index(drop=True)

    return df_merged

# Execute the merge
df_full = merge_weather_and_grid(df_weather_hist, df_grid_hist)
display(df_full.head())

,Date,Hr_End,System_Load,Dry_Bulb,Dew_Point
0,2025-09-10,1,10728.664,58.8,55.9
1,2025-09-10,2,10619.604,58.6,55.6
2,2025-09-10,3,10392.207,58.4,55.3
3,2025-09-10,4,10333.420,58.5,55.2
4,2025-09-10,5,10528.720,58.3,55.1


In [53]:
import sys
from pathlib import Path

# pipeline root must be on sys.path so `src.features...` resolves
sys.path.insert(0, str(Path.home() / "GridCast" / "pipeline"))

import mlflow, pickle
from mlflow.tracking import MlflowClient

# Point to the correct DB (the one in pipeline/)
db_path = Path.home() / "GridCast" / "pipeline" / "mlflow_gridcast.db"
mlflow.set_tracking_uri(f"sqlite:///{db_path}")

client = MlflowClient()
mv = client.get_model_version_by_alias("grid_load_model", "champion")

artifact_uri = f"runs:/{mv.run_id}/preprocessor/preprocessor.pkl"
local_path = mlflow.artifacts.download_artifacts(artifact_uri=artifact_uri)

# now unpickling works because src.* is importable
with open(local_path, "rb") as f:
    preprocessor = pickle.load(f)



In [55]:
TARGET = "System_Load"

X = df_full.drop(columns=[TARGET])
y = df_full[TARGET].values

preds = champion_model.predict(preprocessor.transform(X))

# index-safe assignment
results = df_full.copy()
results.loc[X.index, "Predicted_System_Load"] = preds
results["Error"]        = results["Predicted_System_Load"] - results[TARGET]
results["Abs_Error"]    = results["Error"].abs()
results["APE_%"]        = results["Abs_Error"] / results[TARGET] * 100

mae   = mean_absolute_error(y, preds)
rmse  = np.sqrt(mean_squared_error(y, preds))
r2    = r2_score(y, preds)
mape  = np.mean(np.abs((y - preds) / y)) * 100
smape = np.mean(2 * np.abs(preds - y) / (np.abs(y) + np.abs(preds))) * 100
bias  = np.mean(preds - y)

print("=" * 50)
print(f"MAE   (MW)        : {mae:,.2f}")
print(f"RMSE  (MW)        : {rmse:,.2f}")
print(f"MAPE  (%)         : {mape:.2f}")
print(f"sMAPE (%)         : {smape:.2f}")
print(f"R²                : {r2:.4f}")
print(f"Mean Bias (MW)    : {bias:,.2f}  ({'over' if bias>0 else 'under'}-predicting)")
print(f"Mean Load (MW)    : {np.mean(y):,.2f}")
print(f"1 - MAE/mean      : {(1 - mae/y.mean())*100:.2f}%   (NOT accuracy)")
print("=" * 50)

display(results.head(20))

MAE   (MW)        : 1,335.45
RMSE  (MW)        : 1,915.67
MAPE  (%)         : 12.50
sMAPE (%)         : 11.09
R²                : -1.2077
Mean Bias (MW)    : 1,106.26  (over-predicting)
Mean Load (MW)    : 11,465.59
1 - MAE/mean      : 88.35%   (NOT accuracy)


,Date,Hr_End,System_Load,Dry_Bulb,Dew_Point,Predicted_System_Load,Error,Abs_Error,APE_%
0,2025-09-10,1,10728.664,58.8,55.9,10416.241211,-312.422789,312.422789,2.912038
1,2025-09-10,2,10619.604,58.6,55.6,10081.556641,-538.047359,538.047359,5.066548
2,2025-09-10,3,10392.207,58.4,55.3,9863.577148,-528.629852,528.629852,5.086791
3,2025-09-10,4,10333.420,58.5,55.2,9808.659180,-524.760820,524.760820,5.078288
4,2025-09-10,5,10528.720,58.3,55.1,9958.978516,-569.741484,569.741484,5.411308
5,2025-09-10,6,10979.342,58.2,55.3,10721.674805,-257.667195,257.667195,2.346836
6,2025-09-10,7,11880.971,57.7,55.2,11912.279297,31.308297,31.308297,0.263516
7,2025-09-10,8,12165.776,58.0,55.4,12705.765625,539.989625,539.989625,4.438596
8,2025-09-10,9,11952.447,59.5,56.5,13183.473633,1231.026633,1231.026633,10.299369
9,2025-09-10,10,11298.770,61.2,56.8,13407.397461,2108.627461,2108.627461,18.662451
